# PreDiViD real-data reproduction: Prediction preflight and first replay

This notebook is a **read-only preparation and reproducibility companion** to the original *Prediction Module - COVID vs Monkeypox Observation.ipynb* and *Prediction Module - Full Benchmark.ipynb*. It does not replace either notebook, nor claim that a restored graph dump recreates unavailable observation CSVs.

**Original sequence:** graph + ontologies → six Detection observation tables → top-25 concept selection → binary BoW → K-Means → WHO reference check → fuller embedding and spatiotemporal comparisons → independent evidence construction for Story Forest.


In [ ]:
from pathlib import Path
import subprocess, sys, json
REPO = Path.cwd().resolve()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'pyproject.toml').exists(), 'Open this notebook inside the DDPF repository.'
print('Repository:', REPO)
def run(*args):
    return subprocess.run([sys.executable, *map(str,args)], cwd=REPO, check=False)


## 1. Archive versions and checksums

The repository `PropaPhen.owl` differs bytewise from the published Zenodo `10214515` copy. Keep both; do not overwrite historical results.

In [ ]:
run('scripts/realdata/zenodo_fetch.py','--check')
# Explicitly run this if local ontology files are not yet downloaded:
# run('scripts/realdata/zenodo_fetch.py')
# For the ~2.2 GB dump, manually run:
# run('scripts/realdata/zenodo_fetch.py','--include-dump')


## 2. Inspect ontology + Neo4j schema before event construction

A graph `CUI -> isReported -> Country` WHO reference link is not the same as a per-document story edge. Run only against your **isolated restored database** with credentials in `.env`.

In [ ]:
RUN_GRAPH_AUDIT = False  # Set True only after loading the dump on an isolated Neo4j instance
if RUN_GRAPH_AUDIT:
    run('scripts/realdata/kg_provenance_audit.py','--limit','20')
else:
    print('Skipped Neo4j audit: set RUN_GRAPH_AUDIT = True after restore.')


## 3. Preflight the ORIGINAL Prediction notebooks

The printed notebook cell numbers refer to the originals. The full benchmark needs additional precomputed embeddings and gazetteers.

In [ ]:
run('scripts/realdata/notebook_preflight.py','--mode','all')


## 4. Reproduce the first Prediction branch in notebook order

Original *Observation* notebook, cells 14–20 (load/clean/concept selection), 22–27 (BoW), 30–39 (K-Means/evaluation). This runs only if all six CSV tables are available.

In [ ]:
files = ['covid_observations_journal.csv','covid_observations_medical.csv',
         'covid_observations_social.csv','monkeypox_observations_journal.csv',
         'monkeypox_observations_medical.csv','monkeypox_observations_social.csv']
root = REPO/'data'/'neo4j'
missing = [n for n in files if not (root/n).exists()]
if missing:
    print('NOT READY: recover or regenerate original Detection observations first:')
    print('\n'.join('  '+n for n in missing))
else:
    run('scripts/realdata/replay_bow.py','--data-dir',str(root),
        '--outdir','outputs/realdata/bow_kmeans')


## 5. Return to the original notebooks for the rest

**WHO reference:** observation notebook cells 40–66, read-only queries to `isReported` and temporal filters; review hard-coded credentials.

**Full comparison:** *Prediction Module - Full Benchmark.ipynb*: configuration cells 12–16, methods 18–20, 4 ablations at cells 22/25/28/31. WHO-filtered branch 35–42. **Cell 36 contains Cypher `MERGE`, which writes to the graph. Do not run it on the original archive.**

**New Story Forest:** prepare documented events from real source texts, without treating KG proximity as transmission or causal continuity. See `docs/realdata/REPRODUCTION.md`.
